In [1]:

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

# Sample dataset (sentence + POS tags)
sentences = [
    ["I", "love", "AI"],
    ["I", "love", "coding"]
]

tags = [
    ["PRON", "VERB", "NOUN"],
    ["PRON", "VERB", "NOUN"]
]

# Create word vocabulary
words = list(set(w for s in sentences for w in s))
word2idx = {w: i+1 for i, w in enumerate(words)}

# Create tag vocabulary
tag_list = list(set(t for s in tags for t in s))
tag2idx = {t: i for i, t in enumerate(tag_list)}

# Convert words & tags to numbers
X = [[word2idx[w] for w in s] for s in sentences]
y = [[tag2idx[t] for t in s] for s in tags]

# Padding
X = pad_sequences(X, padding='post')
y = pad_sequences(y, padding='post')

# One-hot encode output
y = np.array([np.eye(len(tag_list))[i] for i in y])

# Build LSTM model
model = Sequential()
model.add(Embedding(input_dim=len(words)+1, output_dim=8))
model.add(LSTM(16, return_sequences=True))
model.add(Dense(len(tag_list), activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
model.fit(X, y, epochs=50, verbose=0)

# Test
test_sentence = ["I", "love", "AI"]
test_seq = [word2idx.get(w, 0) for w in test_sentence]
test_seq = pad_sequences([test_seq], maxlen=X.shape[1], padding='post')

pred = model.predict(test_seq)
pred_tags = [tag_list[np.argmax(p)] for p in pred[0]]

print("Sentence:", test_sentence)
print("Predicted POS Tags:", pred_tags)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 343ms/step
Sentence: ['I', 'love', 'AI']
Predicted POS Tags: ['PRON', 'NOUN', 'NOUN']
